# Studio 3 — your first night in Pyomo 🥫🍔

**OPIM 5641 · Business Decision Modeling · Dr. Dave Wanik, UConn**

So far we've solved LPs three ways: brute force, the graphical method, and Simplex by hand. Tonight
we hand the work to a solver. You write down the model, Pyomo builds it, and the solver does the
Simplex for you.

The plan:

1. **Warm-up:** Flair's Furniture in Pyomo. You already know the answer (320 tables, 360 chairs,
\$4,040), so if the solver agrees, you know your code is right.
2. **Stigler's diet:** 77 foods, 9 nutrients, and the cheapest way to stay alive in 1939.
3. **Your turn:** McDonald's. Feed a football team for a day, as cheaply as possible.
4. **If there's time:** sensitivity analysis. What is one more carpentry hour worth?

The big idea tonight is the **constraint list**. Write a constraint once inside a loop, and it doesn't
care if there are 2 products or 77 foods.

## Import modules (download Pyomo)

You must do this every time! This installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

# Part 1 · Warm-up: Flair's Furniture in Pyomo

Same problem as Studio 2. Flair makes **tables** and **chairs**.

| | Tables | Chairs | Hours available |
|---|---|---|---|
| Carpentry | 3 | 4 | 2,400 |
| Painting | 2 | 1 | 1,000 |
| **Profit (USD)** | 7 | 5 | - |

Plus: **no more than 450 chairs**, and **at least 100 tables**.

$$Max\ Z = 7T + 5C$$

subject to:
* $3T + 4C \le 2400$ `carpentry`
* $2T + 1C \le 1000$ `painting`
* $C \le 450$ `chair limit`
* $T \ge 100$ `table minimum`
* $T, C \ge 0$ `non-negativity`

## 1a · One constraint at a time

This is the general framework: **model → variables → objective → constraints → solve → read the
results.** Every Pyomo model you ever write has these same pieces.

In [ ]:
# you will always need to write this code
model = ConcreteModel()

# declare decision variables
model.tables = Var(domain=NonNegativeReals)
model.chairs = Var(domain=NonNegativeReals)

# declare objective
model.profit = Objective(
                      expr = 7*model.tables + 5*model.chairs, # values come from the table
                      sense = maximize)

# write the constraints one by one, don't forget LHS vs. RHS
# it's OK to have mixed constraints (LHS <= RHS vs. LHS >= RHS)!!!
model.constraint1 = Constraint(expr = 3*model.tables + 4*model.chairs <= 2400) # carpentry hours
model.constraint2 = Constraint(expr = 2*model.tables + 1*model.chairs <= 1000) # painting hours
model.constraint3 = Constraint(expr = model.chairs <= 450) # chair limit
model.constraint4 = Constraint(expr = model.tables >= 100) # table minimum

# show the model you've created
model.pprint()

In [ ]:
#            solver          path the solver
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model).write()

Look for `Termination condition: optimal`. If you see it, you solved! If not, you probably have a
bug (a typo in a constraint, the wrong sense, etc.).

Now read the results. **Don't forget the `()` after everything** - lots of students forget this and
get a bug!

In [ ]:
print("Profit = ", model.profit(), " USD")
print("Tables = ", model.tables(), " units")
print("Chairs = ", model.chairs(), " units")
print()
print("Carpentry = ", model.constraint1(), " hours (of 2400)")
print("Painting = ", model.constraint2(), " hours (of 1000)")
print("Chairs = ", model.constraint3(), " units (limit 450)")
print("Tables = ", model.constraint4(), " units (minimum 100)")

**320 tables, 360 chairs, \$4,040.** Same answer as brute force, the graphical method, and
Simplex. Fourth way, same answer.

Which constraints are **binding** (LHS = RHS)? Carpentry and painting: every hour is used up. That's
the corner where those two lines crossed on your graph.

## 1b · Same model, soft-coded, with a constraint list

Writing constraints one by one is fine for 4 constraints. It's not fine for 400. So we do two things
we already know from brute force:

1. **Soft-code the data** in dictionaries.
2. **Write each kind of constraint once, inside a loop**, and add it to a `ConstraintList()`.

The decision variables become **one indexed variable**, `model.x[...]`, with one entry per product.

In [ ]:
# soft-code the data
products = ["tables", "chairs"]

profit = {
    "tables" : 7,
    "chairs" : 5
}

# hours each product needs in each department
hours = {
    "carpentry" : {"tables" : 3, "chairs" : 4},
    "painting"  : {"tables" : 2, "chairs" : 1}
}

# hours available this month
total_time = {
    "carpentry" : 2400,
    "painting"  : 1000
}

In [ ]:
# declare the model
# ONE indexed variable over products: model.x = Var(products, domain=NonNegativeReals)
# objective: obj_expr = 0, then loop over products and += profit[product] * model.x[product]
# model.constraints = ConstraintList()
# loop over departments: build dept_expr, then model.constraints.add(dept_expr <= total_time[dept])
# then add the chair limit and the table minimum
# model.pprint()
# YOUR CODE HERE 🛠️  (we build this together in studio)


In [ ]:
# solve it with cbc, then print the profit and each product (don't forget the () )
# YOUR CODE HERE 🛠️  (we build this together in studio)


Same answer. But look at what changed: **the constraint code no longer knows how many products or
departments there are.** Add a desk to `products` and a shipping department to `hours`, and the loop
handles it. That's what we need for the next problem, which has 77 products.

# Part 2 · Stigler's diet 🥫

In 1945, the economist **George Stigler** asked a simple question: what is the cheapest diet that
gives a person every nutrient they need for a year? He had **77 foods** and **9 nutrients**, with 1939
prices.

He couldn't solve it exactly (Simplex hadn't been invented yet), so he used clever trial and error
and got **\$39.93 a year**. A couple of years later, it became one of the first big problems ever
solved with the Simplex method: a team of clerks worked it out on desk calculators over many days.
The true optimum was a little lower.

Tonight it takes your laptop less than a second.

**A twist in the data.** Each number in Stigler's table is **how much of a nutrient you get for \$1
of that food** (1939 prices). So the decision variable is **how many dollars per day to spend on each
food**, and the cost of the diet is just the total of those dollars.

$$Min\ Z = \sum_{f} x_f$$

subject to, for every nutrient $n$:

$$\sum_{f} (\text{amount of nutrient } n \text{ per dollar of food } f) \cdot x_f \ge \text{daily minimum}_n$$

$$x_f \ge 0$$

That's a **covering** model: minimize cost, cover every requirement (`>=`). Same family as the trail
mix in M3.

The data is the same as the Google OR-Tools Stigler example. Let's read it from the CSV.

In [ ]:
# read the data
foods_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/stigler_foods.csv')
req_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/stigler_requirements.csv')

print(len(foods_df), "foods")
foods_df.head()

In [ ]:
# the daily minimums - calories are in THOUSANDS (3 = 3,000 calories)
req_df

## Soft-code the data into dictionaries

Same idea as Flair's: one dictionary per piece of the model. Loops only, so you can see every step.

In [ ]:
# the list of foods and the list of nutrients
foods = list(foods_df['food'])
nutrients = list(req_df['nutrient'])

# daily minimum for each nutrient, plus a readable label
minimum = {}
label = {}
for i in range(len(req_df)):
    nutrient = req_df.loc[i, 'nutrient']
    minimum[nutrient] = req_df.loc[i, 'daily_minimum']
    label[nutrient] = req_df.loc[i, 'label']

# content[food][nutrient] = how much of that nutrient you get for $1 of that food
content = {}
for i in range(len(foods_df)):
    food = foods_df.loc[i, 'food']
    content[food] = {}
    for nutrient in nutrients:
        content[food][nutrient] = foods_df.loc[i, nutrient]

print(minimum)
print()
print("Wheat Flour:", content['Wheat Flour (Enriched)'])

## Build the model

Look how short this is. **77 decision variables and 9 constraints**, and the constraint code is the
same shape as Flair's department loop.

In [ ]:
# declare the model
# model.x = Var(foods, domain=NonNegativeReals)   <- dollars per day on each food
# objective: obj_expr = 0, loop over foods and add model.x[food], then sense = minimize
# model.constraints = ConstraintList()
# loop over nutrients: build nutrient_expr from content[food][nutrient] * model.x[food]
#     then model.constraints.add(nutrient_expr >= minimum[nutrient])
# print the number of variables and constraints (skip pprint - it's huge!)
# YOUR CODE HERE 🛠️  (we build this together in studio)


In [ ]:
# solve it with cbc
# YOUR CODE HERE 🛠️  (we build this together in studio)


## Read the results

Same printout as the Google OR-Tools example: what to buy for the year, the annual price, and the
nutrients per day against each minimum.

In [ ]:
# loop over foods: if model.x[food]() > 0.000001, print the food and 365 * its dollars per day
# print the optimal annual price: 365 * model.cost()
# loop over nutrients and print model.constraints[k]() next to the minimum (k starts at 1!)
# YOUR CODE HERE 🛠️  (we build this together in studio)


**\$39.66 a year** - about 11 cents a day. Stigler's trial-and-error guess was \$39.93, so he was
off by about 27 cents. Not bad for doing it by hand!

The winning diet: **wheat flour, beef liver, cabbage, spinach, and navy beans.** For a year.

Which nutrients are **binding** (exactly at the minimum)? Those are the ones driving the cost. The ones
with room to spare (protein, iron...) come along for free.

**Talk with your partner:**
1. Out of 77 foods, the solver picked only 5. Why so few? (Hint: think about corner points. With 9
   constraints, how many variables can be non-zero at a corner?)
2. Would you eat this diet? What's missing from the model?

# Part 3 · Your turn: feed a football team at McDonald's 🏈🍔

The team bus pulls into McDonald's. **53 players**, one day of meals, and the athletic department is
paying. What's the cheapest order that keeps everyone fueled?

Same model as Stigler, with three new wrinkles:

1. **The decision variable is a count:** how many of each menu item to order for the whole team.
2. **Some rules are minimums and some are maximums.** Football players need *at least* 4,000 calories
   and 150 g of protein, but *no more than* 5,000 calories, 150 g of fat, and 5,000 mg of sodium. So
   each nutrient can have a `>=` constraint, a `<=` constraint, or both.
3. **The rules are per player**, so multiply by the team size.

The nutrition facts come from McDonald's published nutrition data (the Kaggle *Nutrition Facts for
McDonald's Menu* dataset). The prices are **approximate** US prices; change them to match your local
McDonald's if you like.

In [ ]:
# read the data
menu_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/mcdonalds_menu.csv')
mreq_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/mcdonalds_requirements.csv')

menu_df

In [ ]:
# rules per player per day - a blank (NaN) means there is no rule in that direction
mreq_df

In [ ]:
# team_size = 53 and max_per_player = 4
# build price[item] and amount[item][nutrient] from menu_df (same loop as Stigler's content dictionary)
# build low[nutrient], high[nutrient] and label[nutrient] from mreq_df
# YOUR CODE HERE 🛠️  (we build this together in studio)


In [ ]:
# declare the model
# model.x = Var(items, domain=NonNegativeReals)   <- how many of each item for the whole team
# objective: total bill = price[item] * model.x[item], summed with a loop, minimize
# model.constraints = ConstraintList()
# loop over nutrients: build nutrient_expr, then
#     if not pd.isna(low[nutrient]):  add nutrient_expr >= low[nutrient] * team_size
#     if not pd.isna(high[nutrient]): add nutrient_expr <= high[nutrient] * team_size
# loop over items: model.x[item] <= max_per_player * team_size
# YOUR CODE HERE 🛠️  (we build this together in studio)


In [ ]:
# solve it with cbc
# YOUR CODE HERE 🛠️  (we build this together in studio)


In [ ]:
# print the total bill and the bill per player
# loop over items: print the ones you order, total and per player
# loop over nutrients: add up amount[item][nutrient] * model.x[item]() and divide by team_size
# YOUR CODE HERE 🛠️  (we build this together in studio)


**Talk with your partner:**
1. Set `max_per_player = 100` and re-run. What does the "cheapest" team diet look like now? (Count
   the apple pies.) How much did the variety rule cost per player?
2. Which nutrition rules are binding? Which ones did the menu cover for free?
3. You can't order 5.7 hamburgers. Change the domain to `NonNegativeIntegers` and re-solve. Did the
   bill go up? (That's integer programming - coming in Module 5.)
4. Ban something you'd never eat, e.g. `model.constraints.add(model.x['Filet-O-Fish'] == 0)`. What
   does your preference cost the team?

# Part 4 · If there's time: sensitivity analysis

Back to Flair's. At the optimum, every carpentry hour is used up. **What would one more carpentry hour
be worth?**

We find out the same way as in the M3 allocation notebook: loop over the carpentry hours, re-solve
each time, and store the results.

In [ ]:
myHours = np.arange(1200, 3300, 100)
print(myHours)

In [ ]:
# myResults = pd.DataFrame()
# for a in myHours:
#     rebuild Flair's model (one constraint at a time is fine), but carpentry is <= a instead of <= 2400
#     solve it, then store [a, profit, tables, chairs] as a row with pd.concat
# rename the columns: Carpentry Hours, Profit, Tables, Chairs
# YOUR CODE HERE 🛠️  (we build this together in studio)


In [ ]:
# add a column: the change in profit divided by the change in carpentry hours
# YOUR CODE HERE 🛠️  (we build this together in studio)


In [ ]:
# plot ProfitDiffOneHour against Carpentry Hours
# YOUR CODE HERE 🛠️  (we build this together in studio)


Read the plot as a staircase:

- **Below 1,500 hours:** carpentry is so scarce we only make tables, and each extra hour is worth
  about **\$2.33** (one more third of a table, at \$7 each).
- **From 1,500 to 2,625 hours:** each extra hour is worth exactly **\$0.60**. That's the **shadow price**
  of carpentry at our optimum.
- **Above 2,625 hours:** we hit the 450-chair limit, and more carpentry is worth **\$0**. If a scarce
  resource becomes 'unlimited', its value becomes ZERO! Amazing.

If you took Flair's all the way to the final Simplex tableau, look at the bottom row under the
carpentry slack: **3/5 = 0.60**. The bottom row of the final tableau *is* the shadow prices.

**On your own:** run the same loop on the painting hours. What is one more painting hour worth? (Check
your final tableau again.)

## Bottom line

- The Pyomo recipe never changes: **model → variables → objective → constraints → solve → read the
  results.**
- A **ConstraintList** plus a loop lets you write a constraint once and apply it to every department,
  every nutrient, every menu item. Two products or 77 foods, same code.
- **Soft-code the data** (dictionaries, or straight from a CSV), and the model doesn't change when the
  data does.
- **Binding** constraints (LHS = RHS) are the ones holding you back, and the sensitivity loop tells
  you exactly what loosening them is worth.

**Save it to GitHub before you leave:** File → Save a copy in GitHub.